KBO 중계 - 규정 매핑기


In [3]:
import pandas as pd
import re

# 1) 파일 로드
rules_master_path = "rules_master.csv"
rules_event_mapping_path = "rules_event_mapping.csv"

cards = pd.read_csv(rules_master_path)
mapping = pd.read_csv(rules_event_mapping_path)

# priority 기준으로 규칙 정렬
PRIORITY_SCORE = {"HIGH": 3, "MEDIUM": 2, "LOW": 1}


# 2) 텍스트 전처리
def normalize_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    return s

# =========================
# 3) event_key별 별칭(키워드) 사전 (이 부분은 llm에 의존하는게 나을수도 있음. 일단 할루시네이션을 줄인다는 목적으로 문자중계 보면서 대체용어 최대한 가져와봤습니다.)
EVENT_ALIASES = {
    # HIT
    "SINGLE": ["단타", "땅볼"],
    "DOUBLE": ["2루타"],
    "TRIPLE": ["3루타"],
    "HOME_RUN": ["홈런", "솔로홈런", "투런홈런", "쓰리런홈런" "HR", "만루홈런", "1점홈런", "2점홈런", "3점홈런"],
    "HIT": ["안타", "히트", "적시타"],

    # OUT
    "STRIKEOUT": ["삼진", "헛스윙 삼진", "루킹 삼진"],
    "GROUND_OUT": ["땅볼", "땅볼 아웃", "땅볼로 아웃", "땅볼 타구"],
    "FLY_OUT": ["뜬공", "플라이", "플라이 아웃", "뜬공 아웃"],
    "DOUBLE_PLAY": ["병살", "병살타", "더블플레이"],
    "FORCE_OUT": ["포스 아웃", "포스아웃"],

    # BASERUN
    "STEAL": ["도루", "도루 성공", "도루 실패"],
    "ADVANCE_BASE": ["진루", "주자 진루", "진루함", "진루했다", "진루합니다"],

    # SCORE/DEFENSE/PLAY
    "RUN_SCORED": ["득점", "홈인", "득점했다", "홈으로 들어와"],
    "ERROR": ["실책", "에러"],
    "SACRIFICE": ["희생타", "희생번트", "희번", "희플", "희생플라이"],

    # COUNT
    "STRIKE": ["스트라이크", "파울 스트라이크", "헛스윙", "루킹"],
    "BALL": ["볼"],
    "FOUL": ["파울"],
    "WALK": ["볼넷", "4구", "사사구"],
    "HBP": ["몸에 맞는 공", "사구", "데드볼", "HBP"],
}

# 4) alias 테이블 생성 (event_key -> rule_key 후보로 연결)
def build_alias_table(mapping_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, r in mapping_df.iterrows():
        event_key = str(r["event_key"]).strip()
        rule_key = str(r["rule_key"]).strip()
        priority = str(r["priority"]).strip().upper()

        aliases = EVENT_ALIASES.get(event_key, [])
        for a in aliases:
            rows.append({
                "alias": a,
                "event_key": event_key,
                "rule_key": rule_key,
                "priority": priority,
                "priority_score": PRIORITY_SCORE.get(priority, 0),
                "alias_len": len(a),
            })

    alias_df = pd.DataFrame(rows)

    # 우선순위: (1) priority 높은 것 (2) alias 길이 긴 것(더 구체적) 먼저
    alias_df = alias_df.sort_values(
        by=["priority_score", "alias_len"],
        ascending=[False, False]
    ).reset_index(drop=True)

    return alias_df

aliases = build_alias_table(mapping)


# 5) 룰카드 조회
cards = cards.copy()
cards["rule_key"] = cards["rule_key"].astype(str).str.strip()
cards_index = cards.set_index("rule_key")

def get_rule_card(rule_key: str) -> dict:
    rule_key = str(rule_key).strip()
    if rule_key not in cards_index.index:
        return None
    row = cards_index.loc[rule_key]
    return {
        "rule_key": rule_key,
        "title": row.get("title", ""),
        "short_def": row.get("short_def", ""),
        "official_rule": row.get("official_rule", ""),
        "exception": row.get("exception", ""),
    }

# 6) event_text -> rule_key 탐지
#    - 가장 간단/안정: alias(키워드) 포함 여부로 매칭
def detect_rule_keys(event_text: str, top_k: int = 3):
    text = normalize_text(event_text)

    hits = []
    for _, r in aliases.iterrows():
        a = r["alias"]
        if a and (a in text):
            hits.append((r["rule_key"], r["event_key"], r["priority_score"], r["alias_len"], a))

    # 1) 아무것도 못 찾으면 None
    if not hits:
        return []

    # cf) "사사구"는 HBP/볼넷 둘 다 걸릴 수 있으니,
    #    문장에 '몸에 맞'/'사구'/'데드볼'이 있으면 HBP 우선
    if "사사구" in text:
        if ("몸에 맞" in text) or ("사구" in text) or ("데드볼" in text) or ("HBP" in text):
            hits = [h for h in hits if h[1] != "WALK"]  # WALK 후보 제거
        else:
            hits = [h for h in hits if h[1] != "HBP"]  # HBP 후보 제거

    # 정렬: priority -> alias 길이(구체성)
    hits = sorted(hits, key=lambda x: (x[2], x[3]), reverse=True)

    # rule_key 중복 제거하며 상위 top_k 반환
    seen = set()
    out = []
    for rule_key, event_key, ps, alen, alias in hits:
        if rule_key in seen:
            continue
        seen.add(rule_key)
        out.append({"rule_key": rule_key, "event_key": event_key, "matched_alias": alias})
        if len(out) >= top_k:
            break

    return out

# 7) 최종: 이벤트 문장 -> (탐지된 룰카드들) 반환
def explain_event_with_rules(event_text: str, top_k: int = 2):
    candidates = detect_rule_keys(event_text, top_k=top_k)
    results = []
    for c in candidates:
        card = get_rule_card(c["rule_key"])
        if card:
            card["matched_alias"] = c["matched_alias"]
            card["event_key"] = c["event_key"]
            results.append(card)
    return results

# =========================
# 8) 테스트 예시 #이건 가상으로 examples df를 만들어봤는데 추후 수정해야할듯
# =========================
examples = [
    "9번 타자 볼넷으로 출루",
    "몸에 맞는 공으로 1루 출루",
    "우중간 2루타!",
    "좌익수 뜬공 아웃",
    "유격수 땅볼 병살타로 이닝 종료",
    "홈런! (투런)",
]

for ex in examples:
    print("====", ex)
    res = explain_event_with_rules(ex, top_k=2)
    for r in res:
        print(f"- 매칭: {r['matched_alias']} → {r['rule_key']} ({r['title']})")
        print(f"  요약: {r['short_def']}")
    print()


==== 9번 타자 볼넷으로 출루
- 매칭: 볼넷 → WALK_DEF (볼넷)
  요약: 볼 4개로 출루
- 매칭: 볼 → BALL_DEF (볼)
  요약: 존을 벗어난 공

==== 몸에 맞는 공으로 1루 출루
- 매칭: 몸에 맞는 공 → HIT_BY_PITCH (몸에 맞는 공)
  요약: 공이 몸에 맞아 출루

==== 우중간 2루타!
- 매칭: 2루타 → HIT_DOUBLE (2루타)
  요약: 안타로 2루까지 진루

==== 좌익수 뜬공 아웃
- 매칭: 뜬공 아웃 → FLY_OUT (플라이 아웃)
  요약: 뜬공을 잡혀 아웃

==== 유격수 땅볼 병살타로 이닝 종료
- 매칭: 병살타 → DOUBLE_PLAY (병살타)
  요약: 한 번에 2아웃 발생
- 매칭: 땅볼 → HIT_SINGLE (단타)
  요약: 타자가 안타로 1루까지 진루

==== 홈런! (투런)
- 매칭: 홈런 → HOME_RUN (홈런)
  요약: 타구가 담장을 넘어 득점

